# Cold Start Recommendation System

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
books = pd.read_json(
    "../data/goodreads_books_children.json.gz",
    lines=True,
    compression="gzip"
)

books = books[["book_id", "title", "description"]].copy()
books["description"] = books["description"].fillna("")

books = books.sample(10000, random_state=42).reset_index(drop=True)
books["text"] = books["title"] + " " + books["description"]

vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
tfidf_matrix = vectorizer.fit_transform(books["text"])

content_similarity = cosine_similarity(tfidf_matrix)
book_index = pd.Series(books.index, index=books["book_id"]).drop_duplicates()

In [ ]:
def cold_start_recommend(book_ids, n=10):
    scores = {}
    for book_id in book_ids:
        if book_id not in book_index:
            continue
        idx = book_index[book_id]
        sim_books = list(enumerate(content_similarity[idx]))
        sim_books = sorted(sim_books, key=lambda x: x[1], reverse=True)[1:6]
        for sim_idx, sim_score in sim_books:
            sim_book_id = books.iloc[sim_idx]["book_id"]
            scores[sim_book_id] = scores.get(sim_book_id, 0) + sim_score
    
    sorted_books = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    results = []
    for book_id, score in sorted_books[:n]:
        title = books[books["book_id"] == book_id]["title"].values
        title = title[0] if len(title) > 0 else "Unknown"
        results.append((book_id, title, round(score, 3)))
    
    return results

In [ ]:
sample_books = list(books["book_id"].iloc[:3])

print("Input Books:")
for b in sample_books:
    print(books[books["book_id"] == b]["title"].values[0])

print("\nRecommendations:\n")

for rec in cold_start_recommend(sample_books):
    print(rec)